# Explore the LNMC Layer 5 Pyramidal Connectivity Whole-Cell Patch-Clamp Dataset (Comprehensive)

## Introduction
This notebook provides a comprehensive exploratory analysis of the **Multineuron Whole-Cell Patch-Clamp Dataset of Layer 5 Pyramidal Synaptic Connectivity in Juvenile Rat Somatosensory Cortex**, produced at the EPFL Laboratory of Neural Microcircuitry (LNMC) and described in Perin, Berger & Markram (2011, PNAS).

Learn more:
- Data Package DOI: [10.71728/senscience.t3x9-1z2t](https://sen.science/doi/10.71728/senscience.t3x9-1z2t/)
- Source article: [10.1073/pnas.1016051108](https://doi.org/10.1073/pnas.1016051108)

As a FAIR² Data Package, it ensures accessibility, interoperability, and AI-readiness. FAIR² datasets follow the MLCommons **Croissant** 🥐 format for machine learning datasets. See the [MLCommons Croissant Format Specification](https://docs.mlcommons.org/croissant/docs/croissant-spec.html).

The dataset provides simultaneous whole-cell patch-clamp electrophysiology from up to **twelve layer 5 pyramidal neurons** recorded at once (a purpose-built “12-patch” setup):
- **47 recording sessions**, one per multineuron experiment
- **150,121 current-clamp acquisition sweeps** across 160 distinct Igor-Pro stimulus protocols
- **482 patched-cell electrode configurations**

Three tidy Parquet record-sets index the collection: `session_metadata` (one row per session), `acquisition_series` (one row per sweep), and `electrode_metadata` (one row per patched cell). This notebook loads them via the Croissant metadata published at the package URL, links every column back to its method step, and renders a multineuron current-clamp viewer — reading the raw voltage waveforms on demand from the canonical **NWB** files. Everything comes from the published `fair2.json`: no local file is needed.

### Install and import required libraries

In [ ]:
# Install mlcroissant (used by patch_fair2) and the loading/plotting/NWB dependencies
# (pyarrow reads the Parquet record sets)
# (fsspec + aiohttp read single NWB files from the package archive over HTTP)
!pip install mlcroissant==1.1.1 numpy==2.4.6 pandas==3.1.1 pyarrow==26.0.1
!pip install matplotlib==3.11.2 h5py==3.16.0 fsspec==2026.9.0 aiohttp==3.14.5

In [ ]:
import json, re, sys, tempfile, time, zipfile
import urllib.request
from collections import defaultdict

import fsspec
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import h5py
except ImportError:
    h5py = None

from IPython.display import Markdown, display

plt.style.use("seaborn-v0_8-whitegrid")

## 1. Data Loading
We use the `mlcroissant` library to read the FAIR² `fair2.json` metadata straight from the Data Package URL, then load every record-set it declares into `pandas` DataFrames — the Parquet files are fetched from the URLs in the metadata's `distribution`. All of them are index tables and fit comfortably in memory (the largest, `acquisition_series`, has 150,121 rows). The **raw voltage waveforms** are not in these tables — they remain in the canonical NWB files (the `nwb_ephys_files.zip` distribution) and are read on demand in the electrophysiology viewer below. Every column is linked back to a specific method step via its `prov:wasGeneratedBy` entry in `fair2.json`.

In [ ]:
# patch_fair2 makes the newer FAIR² export readable by mlcroissant (a stopgap, see its docstring)
PATCH_URL = 'https://raw.githubusercontent.com/senscience/notebooks/main/tools/patch_fair2.py'
patch_folder = tempfile.mkdtemp()
urllib.request.urlretrieve(PATCH_URL, f'{patch_folder}/patch_fair2.py')
sys.path.insert(0, patch_folder)
from patch_fair2 import load_dataset

# The FAIR² Data Package: all metadata and data come from here
FAIR2_URL = 'https://sen.science/doi/10.71728/senscience.t3x9-1z2t/fair2.json'
ds = load_dataset(FAIR2_URL)

# Load every record set declared in the Croissant metadata
dataframes = {}
for rs in ds.metadata.record_sets:
    df = pd.DataFrame(ds.records(rs.uuid))
    df.columns = [col.split('/')[-1] for col in df.columns]
    # mlcroissant loads text columns as bytes — normalise here
    for col in df.select_dtypes(include=['object']).columns:
        df[col] = df[col].apply(lambda x: x.decode('utf-8') if isinstance(x, bytes) else x)
    dataframes[rs.name] = df

session_metadata   = dataframes['session_metadata']
acquisition_series = dataframes['acquisition_series']
electrode_metadata = dataframes['electrode_metadata']
stimulus_series    = dataframes.get('stimulus_series')   # injected current (recorded + reconstructed)

print("Record sets loaded with mlcroissant:")
for name, df in dataframes.items():
    print(f"  {name:<18}: {len(df):>8,} rows x {len(df.columns)} cols")

### 1.1 Data dictionary with method-step provenance
Every field of every record set in `fair2.json` is linked to a specific method step via `prov:wasGeneratedBy`, and the steps are grouped into the sections of `methodSection`. `mlcroissant` does not keep these provenance links, so we read them from the same JSON-LD document and surface the linkage in tabular form.

In [ ]:
# The fair2.json JSON-LD itself, from the same URL, for the provenance links
with urllib.request.urlopen(FAIR2_URL) as response:
    fair2 = json.load(response)

# Method sections and their steps
sections = fair2.get('methodSection', [])
step_section = {step['@id']: section.get('name', '')
                for section in sections for step in section.get('step', [])}
print(f"{len(sections)} method sections with {len(step_section)} steps declared in fair2.json's methodSection")

# A wide table: (record set, field, dataType, method section, → method step)
rows = []
for rs in fair2['recordSet']:
    for f in rs.get('field', []):
        step = f.get('prov:wasGeneratedBy') or {}
        data_type = f.get('dataType', '')
        rows.append({
            'record_set':     rs['name'],
            'field':          f['name'],
            'title':          f.get('dct:title', ''),
            'dataType':       (data_type[0] if isinstance(data_type, list) else data_type).split(':')[-1].split('/')[-1],
            'method_section': step_section.get(step.get('@id'), '—'),
            'method_step':    step.get('rdfs:label', '—'),
        })
dd_df = pd.DataFrame(rows)
display(Markdown(f'### {len(dd_df)} variables across {dd_df.record_set.nunique()} record sets — each linked to a method step'))
display(dd_df)

### 1.2 Dataset Metadata
Inspect the dataset's top-level metadata to confirm name, DOI, and licence.

In [ ]:
metadata = ds.metadata.to_json()
creators = [c.name for c in ds.metadata.creators]

name        = metadata.get('name', 'N/A')
description = metadata.get('description', 'N/A')

print(f"Name       : {name}")
print(f"Version    : {metadata.get('version', 'N/A')}")
print(f"License    : {metadata.get('license', 'N/A')}")
print(f"URL        : {metadata.get('url', 'N/A')}")
print(f"Creators   : {len(creators)}")
for c in creators:
    print(f"    - {c}")
print(f"\nDescription: {description}")

## 2. Data Preparation and Integration
We assemble a per-session integrated summary by aggregating the sweep-level table (`acquisition_series`) and the electrode-level table (`electrode_metadata`) onto the session-level table (`session_metadata`), keyed on the canonical `session_id`.

In [ ]:
# --- Per-session sweep summary --- #
acquisition_series['channel_num'] = pd.to_numeric(acquisition_series['channel_num'], errors='coerce')
sweep_summary = (
    acquisition_series.groupby('session_id')
        .agg(n_sweeps=('acquisition_name', 'size'),
             n_protocols=('protocol', 'nunique'),
             n_channels=('channel_num', lambda s: s[s >= 0].nunique()))
        .reset_index()
)

# --- Per-session electrode count --- #
elec_summary = (electrode_metadata.groupby('session_id')
                .agg(n_cells=('cell_index', 'nunique')).reset_index())

# Merge onto session-level metadata
recording_summary = (session_metadata
                     .merge(sweep_summary, on='session_id', how='left')
                     .merge(elec_summary, on='session_id', how='left'))

# Parse experiment-type tag and experimental date from the session_id
def paradigm(sid):
    m = re.match(r'^\d{6}([A-Za-z]+\d*)', sid)
    t = m.group(1) if m else 'other'
    return 'Plast' if t in ('Plast', 'Plastic') else t
recording_summary['paradigm'] = recording_summary['session_id'].map(paradigm)

display(Markdown('### Recording summary (per session)'))
display(recording_summary[['session_id', 'paradigm', 'n_electrodes', 'n_cells',
                           'n_acquisitions', 'n_sweeps', 'n_protocols', 'n_channels']].head())

## 3. Exploratory Data Analysis (EDA)

### 3.1 Overview of the Integrated Dataset

In [ ]:
print("Recording summary:")
print(f"  {recording_summary.shape[0]} sessions × {recording_summary.shape[1]} fields")
print(f"  paradigm distribution : {dict(recording_summary['paradigm'].value_counts())}")
print(f"  cells per session     : {int(recording_summary['n_electrodes'].min())}–{int(recording_summary['n_electrodes'].max())} "
      f"(median {recording_summary['n_electrodes'].median():.0f})")
print()
print("Sweep-level:")
print(f"  {len(acquisition_series):,} acquisition sweeps across {acquisition_series['session_id'].nunique()} sessions")
print(f"  distinct stimulus protocols : {acquisition_series['protocol'].nunique():,}")
print(f"  distinct sweep identifiers  : {acquisition_series['sweep_id'].nunique():,}")
print()
print("Electrode-level:")
print(f"  {len(electrode_metadata):,} patched-cell electrode configurations across {electrode_metadata['session_id'].nunique()} sessions")

### 3.2 Data Visualization

#### 3.2.1 Recording composition
How the sessions are composed: the number of cells patched simultaneously per session, and the number of acquisition sweeps per session.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

vc = recording_summary['n_electrodes'].value_counts().sort_index()
axes[0].bar(vc.index.astype(int).astype(str), vc.values, color='#4C72B0', edgecolor='white')
axes[0].set_xlabel('Cells patched per session'); axes[0].set_ylabel('# sessions')
axes[0].set_title(f'Simultaneously recorded cells (median {int(recording_summary["n_electrodes"].median())})')

axes[1].hist(recording_summary['n_acquisitions'].dropna(), bins=12, color='#DD8452', edgecolor='white')
axes[1].axvline(recording_summary['n_acquisitions'].median(), color='#4C72B0', ls='--',
                label=f"median {int(recording_summary['n_acquisitions'].median()):,}")
axes[1].set_xlabel('Sweeps per session'); axes[1].set_ylabel('# sessions')
axes[1].set_title('Acquisition volume per session'); axes[1].legend()

plt.tight_layout(); plt.show()

#### 3.2.2 The stimulus repertoire
Sweeps are distributed across 160 distinct Igor-Pro stimulus protocols. Here we show the most-represented protocols by total sweep count.

In [ ]:
proto = (acquisition_series.groupby('protocol')
            .agg(n_sweeps=('acquisition_name', 'size'),
                 n_sessions=('session_id', 'nunique'))
            .sort_values('n_sweeps', ascending=False))

top = proto.head(20)[::-1]
fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(top.index, top['n_sweeps'], color='#8172B3', edgecolor='white')
for y, (n, r) in enumerate(zip(top['n_sweeps'], top['n_sessions'])):
    ax.text(n + top['n_sweeps'].max()*0.01, y, f'{n:,} ({r} sess)', va='center', fontsize=7)
ax.set_xlabel('Sweeps'); ax.set_xlim(0, top['n_sweeps'].max() * 1.2)
ax.set_title(f'Stimulus repertoire — 20 most-represented protocols (of {acquisition_series["protocol"].nunique():,})')
plt.tight_layout(); plt.show()

#### 3.2.3 Electrophysiology viewer — injected current & mean response per cell
The curated NWB files carry the experimenters' own **pre-computed trial-averaged responses** (`igor_source_experiment/Packed_Data/Stimulated_{P}G{R}Avg`): one presynaptic cell is driven with a current-pulse train while the mean membrane-potential response of every simultaneously patched cell is recorded. The NWB file is read over HTTP from the `nwb_ephys_files.zip` distribution declared in `fair2.json` — only the bytes the viewer needs are fetched, not the whole archive. To keep that small, the `stimulus_series` record set tells which sweeps drove the cell, so the viewer reads only the driven cell's averages and a sample of its drive sweeps. We read these averages directly (no sweep-by-sweep playback and no re-computation) and show (A) the **current injected** into the driven cell and (B) the **mean response of each cell**. A genuine monosynaptic connection shows up as a sustained postsynaptic depolarisation.

In [ ]:
# --- Electrophysiology viewer: injected current + mean response per cell --- #
# The curated NWB files carry, per session, the experimenters' pre-computed
# trial-averaged responses:  igor_source_experiment/Packed_Data/Stimulated_{P}G{R}Avg
#   → stimulate presynaptic cell (P-1), record the trial-averaged Vm in cell R.
# We show (A) the current injected into the driven cell and (B) the mean
# response of every simultaneously patched cell — no sweep-by-sweep playback.

EXAMPLE_SESSION = '070521L5_1'          # session with clean averages (data-article Fig. 4)
FS = 5000.0                             # sampling rate (Hz)
PRE_CELL = 5                            # presynaptic (driven) cell to feature
N_SWEEPS = 20                           # drive sweeps averaged for the injected current

AVG_RE  = re.compile(r'^Stimulated_(\d+)G(\d+)Avg$')

# The NWB files are in the package's nwb_ephys_files.zip, whose URL is in fair2.json's distribution.
# The archive is stored uncompressed: HTTP range requests read one NWB file inside it, in place.
NWB_ZIP_URL = next(d.content_url for d in ds.metadata.distribution
                   if d.name == 'nwb_ephys_files.zip')

T0 = time.perf_counter()
def progress(message):
    print(f'[{time.perf_counter() - T0:5.1f} s] {message}', flush=True)

def open_nwb(fname):
    """Open an NWB file inside the remote package zip, without downloading the archive."""
    remote = fsspec.open(NWB_ZIP_URL, block_size=2**20, cache_type='blockcache').open()
    progress(f'Reading the table of contents of {NWB_ZIP_URL.rsplit("/", 1)[-1]} '
             f'({remote.size / 1e9:.1f} GB; only the parts needed are downloaded)...')
    archive = zipfile.ZipFile(remote)
    if fname not in archive.namelist():
        return None
    progress(f'Found {fname} ({archive.getinfo(fname).file_size / 1e6:.0f} MB) among '
             f'{len(archive.namelist())} NWB files: reading it in place')
    return archive.open(fname)

NWB_FILE = open_nwb(EXAMPLE_SESSION + '.nwb') if h5py else None
if NWB_FILE is None:
    print('h5py or the NWB file is not available — skipping the trace viewer.')
else:
    with h5py.File(NWB_FILE, 'r') as h:
        progress('Step 1/3 · Listing the pre-computed trial averages...')
        # ---- (1) pre-computed trial-averaged responses (volts), driven cell only ----
        # List the average names (cheap), then read only those of the driven pipette
        pdp = h.get('igor_source_experiment/Packed_Data')
        avg_names = defaultdict(dict)                 # pipette -> {recorded cell -> dataset name}
        if pdp is not None:
            for k in pdp:
                m = AVG_RE.match(k)
                if m:
                    avg_names[int(m.group(1))][int(m.group(2))] = k

        if PRE_CELL + 1 not in avg_names:
            avail = sorted(p - 1 for p in avg_names)
            PRE_CELL = avail[0] if avail else PRE_CELL
        progress(f'           {sum(map(len, avg_names.values()))} averages for {len(avg_names)} driven cells; '
                 f'reading the {len(avg_names.get(PRE_CELL + 1, {}))} of cell {PRE_CELL}...')
        resp = {R: pdp[k][:] for R, k in avg_names.get(PRE_CELL + 1, {}).items()}

        # keep only physiologically patched cells (rest between -85 and -50 mV)
        def rest_mV(v): return np.median(v[:200]) * 1000.0
        patched = sorted(R for R, v in resp.items() if -85 < rest_mV(v) < -50)
        driven  = [R for R in patched if resp[R].max() > 0]      # cell(s) that spiked
        driven  = driven[0] if driven else PRE_CELL

        # ---- (2) injected current into the driven cell (drive sweeps only) ----
        # The stimulus_series record set lists the session's sweeps, so the NWB file's
        # thousands of stimulus entries need not be scanned: the drive sweeps of cell P are
        # its current channel's protocols ending in _P (e.g. H70S8_5: 8 pulses at 70 Hz into
        # cell 5). We average the first N_SWEEPS of its most-repeated drive protocol — the
        # sweeps repeat the same train, so their mean matches that of all of them.
        stim = h.get('stimulus/presentation')
        cur_ch = 2 * PRE_CELL + 1
        inj, sel, drive_protocol = None, [], None
        progress(f'Step 2/3 · Finding the sweeps that drove cell {PRE_CELL} in stimulus_series...')
        if stim is not None and stimulus_series is not None:
            sweeps = stimulus_series[
                (stimulus_series['session_id'] == EXAMPLE_SESSION)
                & (pd.to_numeric(stimulus_series['channel_num'], errors='coerce') == cur_ch)
                & stimulus_series['protocol'].str.endswith(f'_{PRE_CELL}')]
            if len(sweeps):
                drive_protocol = sweeps['protocol'].value_counts().index[0]
                names = sweeps.loc[sweeps['protocol'] == drive_protocol, 'stimulus_name']
                sel   = [n for n in names if n in stim][:N_SWEEPS]
                progress(f'           {len(sweeps)} drive sweeps on channel {cur_ch}; reading '
                         f'{len(sel)} of protocol {drive_protocol} ({len(names)} recorded)...')
            if sel:
                L = min(stim[n]['data'].shape[0] for n in sel)
                traces = []
                for i, n in enumerate(sel, 1):
                    traces.append(stim[n]['data'][:L])
                    if i % 5 == 0 or i == len(sel):
                        progress(f'           {i}/{len(sel)} sweeps read')
                inj = np.vstack(traces).mean(0) * 1e9  # nA

    progress('Step 3/3 · Plotting...')

    n  = len(resp[patched[0]]); t = np.arange(n) / FS * 1000.0                      # ms
    fig, (axA, axB) = plt.subplots(2, 1, figsize=(11, 8),
                                   gridspec_kw={'height_ratios': [1, 3.2]}, sharex=True)

    # ---- Panel A: injected current ----
    if inj is not None:
        ti = np.arange(len(inj)) / FS * 1000.0
        axA.plot(ti, inj, color='#b5322e', lw=1.0)
        axA.set_ylabel('injected\ncurrent (nA)')
        axA.set_title(f'A  Current injected into the driven cell {driven}  '
                      f'(mean of {len(sel)} sweeps of protocol {drive_protocol})', loc='left', fontsize=10)
    else:
        axA.text(0.5, 0.5, 'no injected-current channel recorded for this cell',
                 ha='center', va='center', transform=axA.transAxes, color='#888')
    for s in ['top', 'right']: axA.spines[s].set_visible(False)

    # ---- Panel B: mean response of every patched cell (stacked) ----
    off = 12.0
    for i, R in enumerate(patched):
        v = (resp[R] - np.median(resp[R][:200])) * 1000.0        # baseline-subtracted mV
        y = i * off
        if R == driven:
            col, lab = '#b5322e', f'cell {R} — driven (mean spike train)'
            v = np.clip(v, -off * 0.4, off * 0.9)                # clip spikes to keep layout
        else:
            col, lab = '#2c5f8a', f'cell {R}'
        axB.plot(t, v + y, lw=0.8, color=col)
        axB.text(-18, y, f'cell {R}', ha='right', va='center', fontsize=8,
                 color=col, fontweight='bold' if R == driven else 'normal')
    axB.set_yticks([]); axB.set_xlabel('time (ms)')
    axB.set_xlim(-40, t.max())
    axB.set_title(f'B  Mean response per simultaneously patched cell '
                  f'({len(patched)} cells) — pre-computed trial averages', loc='left', fontsize=10)
    for s in ['top', 'right', 'left']: axB.spines[s].set_visible(False)
    fig.suptitle(f'Injected current & mean synaptic response · {EXAMPLE_SESSION} · '
                 f'stimulate cell {driven}', y=0.99, fontsize=12)
    plt.tight_layout(); plt.show()
    progress('Done.')

    display(Markdown(
        f'*Curated trial-averaged responses read directly from the NWB file '
        f'(`igor_source_experiment/Packed_Data/Stimulated_*Avg`) — {len(patched)} patched '
        f'cells recorded while cell {driven} was driven with the injected-current train in '
        f'panel A. Postsynaptic cells show their mean membrane-potential response; a genuine '
        f'monosynaptic connection appears as a sustained depolarisation.*'))

#### 3.2.4 Channel usage and paradigm composition
How the collection is composed across the multineuron array: sweeps recorded per channel index, and the mix of experimental paradigms across the 47 sessions.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

chan = acquisition_series[acquisition_series['channel_num'] >= 0]['channel_num'].value_counts().sort_index()
axes[0].bar(chan.index.astype(int), chan.values, color='#55A868', edgecolor='white')
axes[0].set_xlabel('Recording channel (cell index)'); axes[0].set_ylabel('Sweeps')
axes[0].set_title('Sweeps per recording channel')

pc = recording_summary['paradigm'].value_counts()
axes[1].bar(pc.index, pc.values, color='#4C72B0', edgecolor='white')
axes[1].set_ylabel('# sessions'); axes[1].set_title('Session composition by paradigm')
axes[1].tick_params(axis='x', rotation=35)
for x, v in enumerate(pc.values):
    axes[1].text(x, v + 0.3, str(v), ha='center', fontsize=8)

plt.tight_layout(); plt.show()

## 4. Conclusion
This notebook walked through a first exploration of the LNMC layer 5 pyramidal connectivity whole-cell patch-clamp dataset:

- **Loading**: Croissant metadata (`fair2.json`, read from the Data Package URL) drove the ingestion of the tidy index record-sets — 47 per-session `session_metadata` rows, 150,121 per-sweep `acquisition_series` rows, and 482 per-cell `electrode_metadata` rows.
- **Data dictionary with method provenance**: every column in every table is linked to a specific method step (and its section in `methodSection`) via `prov:wasGeneratedBy`.
- **Recording composition**: up to twelve cells patched simultaneously (median 12), with a right-skewed distribution of sweeps per session.
- **Stimulus repertoire**: the distribution of sweeps across the 160 distinct Igor-Pro protocols.
- **Electrophysiology viewer**: simultaneous multineuron traces read on demand from the canonical NWB files.
- **Channel usage and paradigm composition**: sweeps per channel across the array and the mix of experimental paradigms.

Everything shown here is reproducible from the published FAIR² package alone, with no local file: the Croissant `recordSet` entries define the tabular index (with method-step linkage), and the raw signal for every sweep is preserved in the canonical NWB files (archived in `nwb_ephys_files.zip`, read remotely). To extend, consider (a) a rigorous, latency-gated connectivity screen across all directed pairs (controlling for inter-electrode crosstalk), (b) extracting quantitative synaptic features (EPSP amplitude, latency, failure rate), or (c) relating connection probability to intersomatic distance — the central result of Perin et al. (2011).